# 75_tuned_ensemble: 実行制御用マスターノートブック

5モデルすべてを別プロセスでチューニングし（各プロセス終了時にメモリをOSへ返す設計）、
最後に等重み平均する。このノートブックは実行を制御するだけで、重い学習コードは
一切含まない。

| スクリプト | 役割 | 特徴量基盤 | GPU要否 |
|---|---|---|---|
| `00_build_features.py` | 444列(54_用)と441列(基盤モデル用、top150/hire_fixed込み)の両方をキャッシュ | — | 不要 |
| `01_tune_54.py` | `54_`の強化Optuna(150試行×3-fold)→最終学習 | 444列 | 不要 |
| `01_tune_xxxx.py` | `xxxx_v4`分類器の強化Optuna(40試行×3-fold)→7シード×6-fold最終学習+回帰ブレンド | 104列(reference) | 不要 |
| `01_tune_tabpfn_hire_fixed.py` | TabPFNの`n_estimators`等をOptuna探索(25試行) | 441列中hire_fixed(78列) | **推奨** |
| `01_tune_tabpfn_top150.py` | 同上、top150(150列) | 441列中top150 | **推奨** |
| `01_tune_tabicl_hire_fixed.py` | TabICLの`n_estimators`をOptuna探索(12試行) | 441列中hire_fixed(78列) | **推奨** |
| `02_final_ensemble.py` | 上記5本のtuned出力を等重み平均 | — | 不要 |

**GPUランタイムを推奨**（ランタイム→ランタイムのタイプを変更→GPU）。TabPFN/TabICLは
CPUだとサンプル数制限に当たる・極端に遅い可能性がある。54_/xxxx_v4はCPU学習なので
GPUの有無を問わない。

`00`のキャッシュは特徴量基盤(444列/441列)ごとに共有されるので、モデルをさらに
増やしたくなったら`01_tune_XXX.py`を追加するだけでよい。

## 0. セットアップ（Driveマウント・サブプロセスへのパス受け渡し）

Colabの`!`マジックは直前セルのPython変数を自動ではシェルに渡さないため、
`os.environ`経由で明示的に`SCRIPTS_DIR`をエクスポートする。

In [2]:
import os
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
SCRIPTS_DIR = PROJECT_ROOT / "src" / "75_tuned_ensemble"
assert SCRIPTS_DIR.exists(), f"{SCRIPTS_DIR} が見つからない"
os.environ["SCRIPTS_DIR"] = str(SCRIPTS_DIR)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SCRIPTS_DIR  = {SCRIPTS_DIR}")

import torch
print(f"GPU利用可能: {torch.cuda.is_available()}")

Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/jaggle_2026
SCRIPTS_DIR  = /content/drive/MyDrive/jaggle_2026/src/75_tuned_ensemble
GPU利用可能: False


## 1. 特徴量パイプラインの構築（サブプロセス、約15〜20分）

444列(54_用)・441列(基盤モデル用、top150/hire_fixed込み)の両方を1回でキャッシュする。
既にキャッシュがある場合はこのセルをスキップしてよい。

In [5]:
!python "$SCRIPTS_DIR/00_build_features.py"

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）
[2026-08-19 10:08:22] [INFO] === [75_00_build_features] 実験開始 ===
[2026-08-19 10:08:22] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260819
[2026-08-19 10:08:22] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/75_00_build_features_checkpoint.csv
[2026-08-19 10:08:22] [INFO] チェックポイントは未作成（新規実行）
[2026-08-19 10:08:23] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)
[2026-08-19 10:08:23] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)
[2026-08-19 10:08:23] [INFO] 定着率: 0.5647
[2026-08-19 10:08:23] [INFO] Train IDs: 2761, Test IDs: 2502
[2026-08-19 10:08:23] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)
[2026-08-19 10:08:23] [INFO] Test  早期退職者: 0名 / 2502名
[2026-08-19 10:08:23] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）
[2026-08-19 10:08:23] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923
✅ split非依存の基本特徴量関数定義完了
[2026-08-19 10:08:23] [INFO] -----------------------------

## 2. `54_`の強化チューニング（サブプロセス、最重量）

150試行×3-fold ≈ 450回のCatBoost学習。

In [6]:
!python "$SCRIPTS_DIR/01_tune_54.py"

[2026-08-19 10:20:17] [INFO] === [75_01_tune_54] 開始 ===
[2026-08-19 10:20:17] [INFO] キャッシュ読み込み完了: X_full=(2761, 444) X_test=(2502, 444)
キャッシュ読み込み完了: X_full=(2761, 444) X_test=(2502, 444)
[2026-08-19 11:32:11] [INFO] Optuna完了 (4314秒): best_value=0.497951
[2026-08-19 11:32:11] [INFO] best_params={'depth': 5, 'learning_rate': 0.012329001338484864, 'l2_leaf_reg': 2.666014772496592, 'border_count': 189, 'bagging_temperature': 0.5488812381367517, 'random_strength': 0.4226831915239426}
best_value(3-fold CV平均logloss) = 0.497951
best_params = {'depth': 5, 'learning_rate': 0.012329001338484864, 'l2_leaf_reg': 2.666014772496592, 'border_count': 189, 'bagging_temperature': 0.5488812381367517, 'random_strength': 0.4226831915239426}
全件学習用の反復数（3-fold平均best_iteration×1.25） = 880

参考: 54_の元のA_PARAMS_ORIGINAL（25試行・単一ホールドアウト）
{'depth': 4, 'learning_rate': 0.03518359458951149, 'l2_leaf_reg': 2.217690447016724, 'border_count': 218, 'bagging_temperature': 0.6787467566574921, 'random_strength': 1.43849469723

## 3. `xxxx_v4`分類器の強化チューニング（サブプロセス、重量）

40試行×3-fold（探索）+ 7シード×6-fold×2(分類器+回帰器)（最終学習）。text_featuresの処理が
あるため`54_`より1trialあたりのコストが高い。

In [7]:
!python "$SCRIPTS_DIR/01_tune_xxxx.py"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 95.0 MB/s eta 0:00:00:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 77.0 MB/s eta 0:00:00:00:0100:01
[2026-08-19 11:33:50] [INFO] === [75_01_tune_xxxx] 開始 ===
[2026-08-19 11:33:52] [INFO] ------------------------------------------------------------
[2026-08-19 11:33:52] [INFO] データを読み込み中...
[2026-08-19 11:33:55] [INFO] 入社時データ_学習: (2761, 20)
[2026-08-19 11:33:55] [INFO] 入社時データ_予測: (2502, 19)
[2026-08-19 11:33:55] [INFO] 月次データ_学習 : (65754, 29)
[2026-08-19 11:33:55] [INFO] 月次データ_予測 : (60048, 29)
[2026-08-19 11:33:55] [INFO] 月次データ_学習_全期間: (257509, 29)
[2026-08-19 11:33:55] [INFO] ------------------------------------------------------------
[2026-08-19 11:33:55] [INFO] 月次集約を実行中...
[2026-08-19 11:34:46] [INFO] 月次集約_学習: (2761, 79), 月次集約_予測: (2502, 79)
[2026-08-19 11:34:46] [INFO] ------------------------------------------------------------
[2026-08-19 11:34:46] [INFO] 入社時メモの文書分割を実

## 4. TabPFN(hire_fixed)のチューニング（サブプロセス、GPU推奨）

探索空間は`n_estimators`/`softmax_temperature`等のみ（25試行）。1回の学習が速いので軽量。

In [3]:
!python "$SCRIPTS_DIR/01_tune_tabpfn_hire_fixed.py"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.3/504.3 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
transforme

## 5. TabPFN(top150)のチューニング（サブプロセス、GPU推奨）

In [4]:
!python "$SCRIPTS_DIR/01_tune_tabpfn_top150.py"

[2026-08-20 00:42:39] [INFO] === [75_01_tune_tabpfn_top150] 開始 ===
tabpfn 2.2.1 / device = cpu
⚠️ GPUが無い。CPU向けに探索を軽量化する（試行数・n_estimators上限を下げる）。
top150: 150列 / 学習2208名 / 検証535名 / 全件2761名
タイミングプローブ: n_estimators=1で 141秒（実際のtrialはn_estimatorsに比例して長くなる。目安: 1trial ≈ 424秒、8試行で合計 ≈ 57分）
[2026-08-20 01:34:28] [INFO] Optuna完了 (2968秒): best_value=0.498142
[2026-08-20 01:34:28] [INFO] best_params={'n_estimators': 4, 'softmax_temperature': 0.7912291401980419, 'balance_probabilities': True, 'average_before_softmax': False}
best_value(holdout logloss) = 0.498142
best_params = {'n_estimators': 4, 'softmax_temperature': 0.7912291401980419, 'balance_probabilities': True, 'average_before_softmax': False}
参考: 63_のデフォルト設定(n_estimators等はライブラリ既定値のまま)でのTabPFN(top150) val ≈ 0.507569（列数が違うため直接比較にはならない）
[2026-08-20 01:48:43] [INFO]   seed=42: 完了 (855秒)
[2026-08-20 02:03:05] [INFO]   seed=2024: 完了 (862秒)
[2026-08-20 02:17:13] [INFO]   seed=7: 完了 (848秒)
[2026-08-20 02:17:13] [INFO] 提出ファイルを保存: /content/drive/MyDr

## 6. TabICL(hire_fixed)のチューニング（サブプロセス、GPU推奨）

探索空間は`n_estimators`のみ（12試行、既定8の2倍=16まで。メモリ制約に配慮して狭めにしてある）。

In [5]:
!python "$SCRIPTS_DIR/01_tune_tabicl_hire_fixed.py"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.9/252.9 kB 11.6 MB/s eta 0:00:00
[2026-08-20 02:28:30] [INFO] === [75_01_tune_tabicl_hire_fixed] 開始 ===
device = cpu
⚠️ GPUが無い。CPU向けに探索を軽量化する（試行数・n_estimators上限を下げる）。TabICLはCPUだとTabPFNよりさらに遅い可能性がある点に注意。
hire_fixed: 78列 / 学習2208名 / 検証535名 / 全件2761名
INFO: You are downloading 'tabicl-classifier-v2-20260212.ckpt', the latest best-performing version, used in our TabICLv2 paper.

Checkpoint 'tabicl-classifier-v2-20260212.ckpt' not cached.

tabicl-classifier-v2-20260212.ckpt: 100% 110M/110M [00:01<00:00, 92.5MB/s]
タイミングプローブ: n_estimators=2で 19秒（目安: 1trial ≈ 93秒、6試行で合計 ≈ 9分）
[2026-08-20 02:33:45] [INFO] Optuna完了 (297秒): best_value=0.511325
[2026-08-20 02:33:45] [INFO] best_params={'n_estimators': 10}
best_value(holdout logloss) = 0.511325
best_params = {'n_estimators': 10}
参考: 70_の既定 n_estimators=8, batch_size=1（メモリ55%減・時間45%減のため既定から下げた値）
[2026-08-20 02:36:15] [INFO]   seed=42: 完了 (150秒)
[2026-08-20 02:38:45] [INFO]   seed=2024: 完了 (150秒)
[2026-0

## 7. 最終アンサンブル（サブプロセス、軽量・数秒〜数十秒）

In [6]:
!python "$SCRIPTS_DIR/02_final_ensemble.py"

[2026-08-20 02:41:21] [INFO] === [75_02_final_ensemble] 開始 ===
【再利用・複製】54_tuned: 20260819_75_01_tune_54_54tuned_testpreds.npy → 20260820_75_02_final_ensemble_input_54_tuned.npy
[2026-08-20 02:41:23] [INFO] 【再利用・複製】54_tuned: 20260819_75_01_tune_54_54tuned_testpreds.npy → 20260820_75_02_final_ensemble_input_54_tuned.npy
【再利用・複製】xxxx_tuned: 20260819_75_01_tune_xxxx_xxxx_tuned_testpreds.npy → 20260820_75_02_final_ensemble_input_xxxx_tuned.npy
[2026-08-20 02:41:23] [INFO] 【再利用・複製】xxxx_tuned: 20260819_75_01_tune_xxxx_xxxx_tuned_testpreds.npy → 20260820_75_02_final_ensemble_input_xxxx_tuned.npy
【再利用・複製】TabPFN(hire_fixed)_tuned: 20260819_75_01_tune_tabpfn_hire_fixed_tabpfn_hire_fixed_tuned_testpreds.npy → 20260820_75_02_final_ensemble_input_TabPFN_hire_fixed_tuned.npy
[2026-08-20 02:41:23] [INFO] 【再利用・複製】TabPFN(hire_fixed)_tuned: 20260819_75_01_tune_tabpfn_hire_fixed_tabpfn_hire_fixed_tuned_testpreds.npy → 20260820_75_02_final_ensemble_input_TabPFN_hire_fixed_tuned.npy
【再利用・複製】TabICL(hire_fixe

## 8. `xxxx_tuned` × 現最良プールのブレンド（サブプロセス、軽量・2026-08-20追記）

現最良ブレンド（`xxxx_v4` × 現最良プール、Public 0.497608）の`xxxx_v4`部分を、
`xxxx_v4`自身を上回った`xxxx_tuned`（0.506457、第3節の出力）に差し替えるだけ。
重みは元のレシピと同じ非チューニングの`w=0.5`のまま（[[ensemble-oof-overfitting]]の
教訓通り重み探索はしない）。第3節を実行済みであること（`01_tune_xxxx.py`の出力が必要）。

In [7]:
!python "$SCRIPTS_DIR/03_xxxx_tuned_pool_blend.py"

[2026-08-20 02:45:52] [INFO] === [75_03_xxxx_tuned_pool_blend] 開始 ===
【再利用・複製】xxxx_tuned(Public 0.506457): 20260819_75_01_tune_xxxx_xxxx_tuned_testpreds.npy → 20260820_75_03_xxxx_tuned_pool_blend_input_xxxx_tuned.npy
[2026-08-20 02:45:52] [INFO] 【再利用・複製】xxxx_tuned(Public 0.506457): 20260819_75_01_tune_xxxx_xxxx_tuned_testpreds.npy → 20260820_75_03_xxxx_tuned_pool_blend_input_xxxx_tuned.npy
【再利用・複製】現最良プール(Public 0.508699): 20260816_pool_top150_hire_fixed_avg.csv → 20260820_75_03_xxxx_tuned_pool_blend_input_現最良プール.csv
[2026-08-20 02:45:52] [INFO] 【再利用・複製】現最良プール(Public 0.508699): 20260816_pool_top150_hire_fixed_avg.csv → 20260820_75_03_xxxx_tuned_pool_blend_input_現最良プール.csv

corr(xxxx_tuned, 現最良プール) = 0.9107  MAD = 0.08828（ノイズ床0.02122）
[2026-08-20 02:45:52] [INFO] xxxx_tuned×プールブレンドを保存: /content/drive/MyDrive/jaggle_2026/data/output/20260820/20260820_75_03_xxxx_tuned_pool_blend_xxxx_tuned_pool_blend_w50.csv

保存: /content/drive/MyDrive/jaggle_2026/data/output/20260820/20260820_75_03_xxxx_t